# 02 Validation analysis

Reads **selected completed** OOF runs under `artifacts/runs/`.
Do **not** tune on Public LB. Fill `SELECTED` after Checkpoint B runs exist.


In [ ]:
from pathlib import Path
import json
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.metrics import roc_auc_score

from smartphone_addiction.paths import project_root

ROOT = project_root()
RUNS = ROOT / "artifacts" / "runs"
FIG = ROOT / "reports" / "figures"
FIG.mkdir(parents=True, exist_ok=True)

# Edit after training completes, e.g. ["20260807T120000Z-catboost-smoke-abcd123"]
SELECTED: list[str] = []
print("selected", SELECTED)
print("available", sorted(p.name for p in RUNS.glob("*") if p.is_dir())[:20])


In [ ]:
rows = []
for run_id in SELECTED:
    run = RUNS / run_id
    metrics = json.loads((run / "metrics.json").read_text(encoding="utf-8"))
    oof = pd.read_parquet(run / "oof_predictions.parquet")
    auc = float(roc_auc_score(oof["addicted_label"], oof["prediction"]))
    rows.append({
        "run_id": run_id,
        "model": metrics.get("model_name"),
        "oof_auc_metrics": metrics.get("oof_auc"),
        "oof_auc_recomputed": auc,
        "oof_coverage": metrics.get("oof_coverage"),
        "n": len(oof),
    })
summary = pd.DataFrame(rows)
display(summary)
if not summary.empty:
    summary.to_csv(FIG / "validation_selected_runs.csv", index=False)


In [ ]:
# Fold / seed comparison when fold_metrics.csv exists
for run_id in SELECTED:
    path = RUNS / run_id / "fold_metrics.csv"
    if not path.is_file():
        print("missing", path)
        continue
    folds = pd.read_csv(path)
    display(run_id, folds.groupby("seed")["auc"].agg(["mean", "std", "min", "max"]))
    ax = folds.boxplot(column="auc", by="seed", figsize=(6, 4))
    plt.suptitle("")
    plt.title(f"{run_id} fold AUC by seed")
    plt.tight_layout()
    plt.savefig(FIG / f"validation_folds_{run_id}.png", dpi=120)
    plt.close()


In [ ]:
# Optional: pairwise OOF correlation when >= 2 runs selected
if len(SELECTED) >= 2:
    first = pd.read_parquet(RUNS / SELECTED[0] / "oof_predictions.parquet")
    second = pd.read_parquet(RUNS / SELECTED[1] / "oof_predictions.parquet")
    merged = first[["id", "prediction"]].merge(
        second[["id", "prediction"]], on="id", suffixes=("_a", "_b")
    )
    corr = merged["prediction_a"].corr(merged["prediction_b"])
    print("oof_pred_corr", corr)
else:
    print("Select >=2 runs to compute OOF prediction correlation.")


## Why not Public LB for tuning?

Public LB is delayed, noisy, and encourages leakage into model selection.
Keep Optuna, ablations, and blend weights on local OOF only; use LB only as a
sanity check after a frozen candidate is ready.
